# Module 3 — Debugging & Failure Modes

> **Time:** 20 minutes.
>
> **What you'll do:** find and fix **4 planted bugs** in your Module 2 system. One bug per failure family from the slides, plus one silent failure.

The system below *was* working at the end of Module 2. Then someone made a few "small changes" and now it misbehaves. Your job: reproduce each issue, classify which failure family it belongs to, and fix it with a **minimal diff**.

**Bug families to expect** (from the slides):

| #   | Family                      | Symptom shape                         |
|-----|-----------------------------|---------------------------------------|
| I   | Hallucination / wrong tool  | Confident but off-policy output       |
| II  | Broken coordination         | Wrong branch, infinite loop, crash    |
| III | State propagation           | Empty / wrong keys, silent corruption |
| —   | Silent failure              | Looks fine but isn't                  |

**Recommended workflow:** run the cells top-to-bottom, observe what breaks, then jump to the matching debugging section below.

## 1.  Setup (same as Module 2)

In [ ]:
%pip install -q \
    langgraph==0.2.* \
    langchain==0.3.* \
    langchain-openai==0.2.* \
    pydantic==2.*


import os

def _ensure_key(name: str, optional: bool = False) -> None:
    """Load an API key from (in order): existing env, Colab Secrets, or getpass.

    Colab Secrets are the recommended path for this workshop — set them ONCE
    via the 🔑 key icon in Colab's left sidebar and every notebook will pick
    them up automatically.
    """
    if os.environ.get(name):
        print(f"  ✓  {name} already set in environment")
        return
    # 1. Try Colab Secrets
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f"  ✓  {name} loaded from Colab Secrets")
            return
    except Exception:
        pass
    # 2. Fallback — prompt the user
    import getpass
    val = getpass.getpass(f"Paste your {name}{' (optional)' if optional else ''}: ").strip()
    if val:
        os.environ[name] = val
        print(f"  ✓  {name} set")
    elif optional:
        print(f"  •  {name} skipped (optional)")
    else:
        print(f"  ⚠   {name} skipped — you'll hit errors later without it")

_ensure_key("OPENAI_API_KEY")

print("Ready.")


## 2.  Knowledge base

In [ ]:
KB = {
    "billing": [
        {"id": "B-001", "title": "Billing cycle and prorations",
         "text": "We bill on the same day each month. If you change plans mid-cycle, the next invoice is prorated."},
        {"id": "B-002", "title": "Refund policy",
         "text": "Refunds are available within 14 days. Issued to original payment method, settle in 5 business days."},
        {"id": "B-003", "title": "Failed payments",
         "text": "Failed payments retry once a day for 3 days, then 7-day grace period."},
    ],
    "technical": [
        {"id": "T-001", "title": "Login issues",
         "text": "Clear cookies for the domain. For MFA failures, check spam and verify registered phone."},
        {"id": "T-002", "title": "API rate limits",
         "text": "Standard plan: 60 req/min. Enterprise: 600 req/min. Respect Retry-After header."},
        {"id": "T-003", "title": "Export failures",
         "text": "Retry from the same dialog — export jobs are idempotent."},
    ],
    "account": [
        {"id": "A-001", "title": "Password reset",
         "text": "Use Forgot password. Reset email valid for 30 minutes. Check spam."},
        {"id": "A-002", "title": "Account closure",
         "text": "Settings → Account → Close. 30-day pending-deletion window, then permanent deletion."},
        {"id": "A-003", "title": "Ownership transfer",
         "text": "Owner invites new owner as Admin, both confirm via email."},
    ],
}

## 3.  State schema (unchanged from M2)

In [ ]:
from typing import TypedDict, Annotated, Literal
from operator import add


class TriageState(TypedDict):
    ticket: str
    category: Literal["billing", "technical", "account"] | None
    urgency:  Literal["low", "med", "high"] | None
    retrieved: list[dict]
    draft: str
    verdict: Literal["pass", "revise"] | None
    revision_count: int
    revisions: Annotated[list[str], add]


def make_initial_state(ticket: str) -> TriageState:
    return {
        "ticket": ticket, "category": None, "urgency": None,
        "retrieved": [], "draft": "", "verdict": None,
        "revision_count": 0, "revisions": [],
    }

## 4.  LLM

In [ ]:
# 🐛  BUG ZONE — somewhere in the cells below there are 4 issues.
from langchain_openai import ChatOpenAI

# Bug #4 lives here. Same code as Module 2, ALMOST.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

## 5.  The four agents and graph (BUGGY)

In [ ]:
import json

def parse_json(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text.strip())


CLASSIFY_PROMPT = """\
Categorize the ticket as one of: billing, technical, account.
Rate urgency as: low, med, high.
Return JSON: {{"category": "...", "urgency": "..."}}

TICKET:
{ticket}
"""


# This node is clean — no bug here.
def classify(state: TriageState) -> dict:
    prompt = CLASSIFY_PROMPT.format(ticket=state["ticket"])
    response = llm.invoke(prompt)
    parsed = parse_json(response.content)
    return {"category": parsed["category"], "urgency": parsed["urgency"]}

In [ ]:
# 🐛  Bug somewhere below.
def retrieve(state: TriageState) -> dict:
    category = state["category"]
    docs = KB.get(category, [])
    # ← Look closely at the key being written.
    return {"docs": docs}

In [ ]:
# 🐛  Bug somewhere below.
DRAFT_PROMPT = """\
You are a customer support agent.

Reply to this ticket. Be helpful and concise.

TICKET:
{ticket}
"""


def draft(state: TriageState) -> dict:
    # ← Notice anything missing in the prompt above?
    context = "\n\n".join(
        f"[{d['id']}] {d['title']}\n{d['text']}"
        for d in state["retrieved"]
    )
    prompt = DRAFT_PROMPT.format(ticket=state["ticket"])
    response = llm.invoke(prompt)
    return {
        "draft": response.content,
        "revisions": [response.content],
    }

In [ ]:
QA_PROMPT = """\
Decide if the DRAFT is acceptable.
Return JSON: {{"verdict": "pass" | "revise"}}

TICKET:
{ticket}

DRAFT:
{draft}
"""


# This node is clean — no bug here.
def qa(state: TriageState) -> dict:
    prompt = QA_PROMPT.format(ticket=state["ticket"], draft=state["draft"])
    response = llm.invoke(prompt)
    parsed = parse_json(response.content)
    return {
        "verdict": parsed["verdict"],
        "revision_count": state["revision_count"] + 1,
    }

In [ ]:
from langgraph.graph import StateGraph, END

MAX_REVISIONS = 2


# 🐛  Bug somewhere below.
def route_qa(state: TriageState) -> str:
    # ← What does this function ACTUALLY return? What does add_conditional_edges expect?
    return state["verdict"]


graph = StateGraph(TriageState)
graph.add_node("classify", classify)
graph.add_node("retrieve", retrieve)
graph.add_node("drafter",  draft)
graph.add_node("qa",       qa)

graph.set_entry_point("classify")
graph.add_edge("classify", "retrieve")
graph.add_edge("retrieve", "drafter")
graph.add_edge("drafter",  "qa")

graph.add_conditional_edges(
    "qa", route_qa,
    {"drafter": "drafter", "END": END},
)

app = graph.compile()
print("Graph compiled (but probably broken).")

## 6.  Reproduce the failures

Try running the graph. Use `stream()` so you can see each node's output and pinpoint where things go wrong.

> **Expected on first run:** something blows up. Read the error carefully — Module 3 slide 7 says "reproduce" is the most-skipped step.

In [ ]:
sample = make_initial_state(
    "Hi, my monthly bill is $50 higher than last month. Can you check what happened?"
)

try:
    for step in app.stream(sample):
        for node, update in step.items():
            print(f"→  {node}")
            for k, v in update.items():
                preview = repr(v)
                if len(preview) > 80:
                    preview = preview[:77] + "..."
                print(f"     {k} = {preview}")
        print()
except Exception as e:
    print(f"\n💥  CRASH: {type(e).__name__}: {e}")

## 7.  Find and fix the bugs

Below are 4 stub sections — one per bug. For each:

1. **Find** — look at the relevant cell above and identify what's wrong
2. **Classify** — which failure family is this? (`I` / `II` / `III` / `silent`)
3. **Fix** — edit the cell above with a minimal diff, then re-run from there

The numbering below is just a suggested order. You may discover them in a different order — that's fine.

### 🐛  Bug A — Crash on first run

**Symptom:** the graph blows up at the QA → next-node edge. Stack trace mentions a `ValueError` or similar about a mapping key.

**Family?** Fill in below — and patch the cell above that owns this bug.

In [ ]:
# TODO — note your diagnosis here, then edit the actual buggy cell above.

DIAGNOSIS = {
    "bug": "A",
    "family": "...",   # 'I' / 'II' / 'III' / 'silent'
    "what_was_wrong": "...",
    "what_i_fixed": "...",
}
print(DIAGNOSIS)

### 🐛  Bug B — Drafts look generic / hallucinated

**Symptom:** after fixing the crash, the draft replies look plausible but don't actually reference the policies in our KB. Run a billing ticket — the draft should mention prorations, but doesn't.

**Family?** Two possible places to look. Check Retriever AND Drafter.

In [ ]:
DIAGNOSIS = {
    "bug": "B",
    "family": "...",
    "what_was_wrong": "...",
    "what_i_fixed": "...",
}
print(DIAGNOSIS)

### 🐛  Bug C — Drafts *still* don't use the policies

**Symptom:** even after fixing the state-key mismatch, the drafts still don't ground in the retrieved policies. The data is in state, but the LLM doesn't see it.

**Family?** Look at the prompt template, not just the function body.

In [ ]:
DIAGNOSIS = {
    "bug": "C",
    "family": "...",
    "what_was_wrong": "...",
    "what_i_fixed": "...",
}
print(DIAGNOSIS)

### 🐛  Bug D — Different runs give different drafts

**Symptom:** rerun the same ticket twice. The drafts come back materially different. Production reproducibility is shot — eval (Module 5) will be useless.

**Family?** Silent failure. The graph works; the *behavior* doesn't.

In [ ]:
DIAGNOSIS = {
    "bug": "D",
    "family": "...",
    "what_was_wrong": "...",
    "what_i_fixed": "...",
}
print(DIAGNOSIS)

## 8.  Verify all four are fixed

Run the graph on three representative tickets. Each should:
- Complete without crashing
- Produce a draft that references specific KB content
- Produce the *same* draft when rerun (because we pinned `temperature=0`)

In [ ]:
verification_tickets = [
    "Hi, my monthly bill is $50 higher than last month. Can you check what happened?",
    "I keep getting 429 errors from your API on the Standard plan.",
    "Forgot my password and the reset email never showed up. Demo in 30 minutes.",
]

for t in verification_tickets:
    try:
        result = app.invoke(make_initial_state(t))
        ok = result["verdict"] is not None and len(result["draft"]) > 50
        print(f"  {'✓' if ok else '✗'}  category={result['category']:9}  "
              f"verdict={result['verdict']:7}  revisions={result['revision_count']}")
    except Exception as e:
        print(f"  ✗  CRASH: {type(e).__name__}: {e}")

## Wrap up

You used Module 3's debugging loop end-to-end:

1. **Reproduced** — let the system actually fail
2. **Isolated** — used `stream()` to see which node's output went wrong
3. **Hypothesized** — matched the symptom to a failure family
4. **Fixed** — minimal diff per bug
5. **Verified** — reran the whole suite to catch regressions

**Up next:** Module 4 — Observability. Instead of `stream()` and print, we wire up LangSmith — and the same investigations get a lot faster.